In [ ]:
# %% [markdown]
# # Weather + Electricity Time-Series Analysis
# Using Open-Meteo ERA5 reanalysis + electricity production/price data  
# (for Norway: Oslo, Kristiansand, Trondheim, Tromsø, Bergen)

# %% [markdown]
## 1. Setup: imports & city data

# %% code
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import dct, idct
from sklearn.neighbors import LocalOutlierFactor
from statsmodels.tsa.seasonal import STL
import requests

# %% code
# Define the five price-areas / cities with geolocation
cities = {
    "NO1": ["Oslo", 59.91, 10.75],
    "NO2": ["Kristiansand", 58.15, 7.99],
    "NO3": ["Trondheim", 63.43, 10.39],
    "NO4": ["Tromsø", 69.65, 18.96],
    "NO5": ["Bergen", 60.39, 5.32],
}
df_cities = pd.DataFrame(
    [(code, name, lat, lon) for code,(name,lat,lon) in cities.items()],
    columns=["price_area","city","latitude","longitude"]
)
display(df_cities)

# %% [markdown]
## 2. Function to download weather data from Open-Meteo

# %% code
def fetch_weather(lat, lon, year,
                  hourly_vars=None,
                  start_date=None, end_date=None,
                  timezone="UTC"):
    """
    Downloads hourly weather data for given lat/lon and year from Open-Meteo “archive” (ERA5) endpoint.
    Returns a pandas DataFrame with at least ‘time’ column + the requested variables.
    """
    if hourly_vars is None:
        hourly_vars = ["temperature_2m","precipitation"]
    if start_date is None:
        start_date = f"{year}-01-01"
    if end_date is None:
        end_date = f"{year}-12-31"
    url = (
        f"https://archive-api.open-meteo.com/v1/era5?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly={','.join(hourly_vars)}"
        f"&timezone={timezone}"
    )
    print("Fetching:", url)
    resp = requests.get(url)
    resp.raise_for_status()
    js = resp.json()
    # convert to DataFrame
    df = pd.DataFrame({
        "time": js["hourly"]["time"]
    })
    for var in hourly_vars:
        df[var] = js["hourly"].get(var)
    # convert time to datetime
    df["time"] = pd.to_datetime(df["time"])
    return df

# %% [markdown]
### Example: fetch for Bergen in 2019

# %% code
df_bergen_2019 = fetch_weather(lat=60.39, lon=5.32, year=2019,
                               hourly_vars=["temperature_2m","precipitation"])
df_bergen_2019.head()

# %% [markdown]
## 3. Temperature outlier detection (DCT + SPC boundaries)

# %% code
def detect_temp_outliers(df, temp_col="temperature_2m",
                         freq_cutoff=365*24//2,  # example: half‐year cut-off
                         n_std=3):
    """
    Takes df with hourly temp, applies DCT high-pass for season removal,
    computes residuals, computes robust mean & std, returns indices of outliers.
    """
    # ensure sorted
    df = df.sort_values("time").reset_index(drop=True)
    temps = df[temp_col].values
    # DCT
    coeffs = dct(temps, norm='ortho')
    # zero out low-frequency components (seasonal) up to cutoff
    coeffs[:freq_cutoff] = 0
    # reconstruct
    residual = idct(coeffs, norm='ortho')
    # robust statistics (median + MAD)
    med = np.median(residual)
    mad = np.median(np.abs(residual-med))
    robust_std = mad * 1.4826
    upper = med + n_std * robust_std
    lower = med - n_std * robust_std
    outlier_mask = (residual > upper) | (residual < lower)
    return {
        "residual": residual,
        "outlier_mask": outlier_mask,
        "upper": upper,
        "lower": lower,
        "median": med,
        "robust_std": robust_std,
        "n_outliers": outlier_mask.sum()
    }

# %% [markdown]
### Plotting the temperature and marking the outliers

# %% code
res = detect_temp_outliers(df_bergen_2019, freq_cutoff=24*30, n_std=3)
plt.figure(figsize=(14,4))
plt.plot(df_bergen_2019["time"], df_bergen_2019["temperature_2m"], label="Temperature")
plt.scatter(df_bergen_2019["time"][res["outlier_mask"]],
            df_bergen_2019["temperature_2m"][res["outlier_mask"]],
            color="red", label="Outliers")
plt.hlines([res["upper"], res["lower"]], xmin=df_bergen_2019["time"].min(),
           xmax=df_bergen_2019["time"].max(), colors="grey", linestyles="--",
           label="SPC boundaries")
plt.legend()
plt.title("Temperature with Outliers – Bergen 2019")
plt.show()

# %% [markdown]
## 4. Precipitation anomaly detection using LOF

# %% code
def detect_precip_anomalies(df, precip_col="precipitation", outlier_prop=0.01):
    """
    Applies LOF to precipitation time‐series, flags the top outlier_prop fraction as anomalies.
    Returns mask of anomalies and LOF scores.
    """
    X = df[[precip_col]].values
    # fewer neighbours if few points
    n_neighbors = min(20, max(5, int(len(X)*0.01)))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=outlier_prop)
    y_pred = lof.fit_predict(X)
    anomaly_mask = (y_pred == -1)
    return {
        "anomaly_mask": anomaly_mask,
        "scores": -lof.negative_outlier_factor_,
        "n_anomalies": anomaly_mask.sum()
    }

# %% [markdown]
### Plot precipitation and mark anomalies

# %% code
pa = detect_precip_anomalies(df_bergen_2019, precip_col="precipitation", outlier_prop=0.01)
plt.figure(figsize=(14,4))
plt.plot(df_bergen_2019["time"], df_bergen_2019["precipitation"], label="Precipitation")
plt.scatter(df_bergen_2019["time"][pa["anomaly_mask"]],
            df_bergen_2019["precipitation"][pa["anomaly_mask"]],
            color="red", label="Anomalies")
plt.legend()
plt.title("Precipitation Anomalies – Bergen 2019")
plt.show()

# %% [markdown]
## 5. STL decomposition of electricity production data from Elhub

# %% [markdown]
```python
def plot_stl(series, period=24*7, seasonal=7, trend=13, robust=True,
             title="STL Decomposition"):
    """
    series: pandas Series with datetime index
    period: seasonal period
    seasonal: smoother length
    trend: smoother length
    robust: boolean
    """
    stl = STL(series, period=period, seasonal=seasonal, trend=trend, robust=robust)
    res = stl.fit()
    fig = res.plot()
    fig.suptitle(title)
    return res
